# Part B — Capacity reconciliation

Portable executable answers for B1–B4. Detailed prose is in `Part_B_readme.md`.

In [1]:
from pathlib import Path
import sys
from IPython.display import Markdown, display

cwd = Path.cwd()
ROOT = cwd if (cwd / 'bench_metrics.py').exists() else cwd / 'partB'
assert (ROOT / 'bench_metrics.py').exists(), 'Run from partB/ or repository root'
sys.path.insert(0, str(ROOT))
from bench_metrics import b1_summary, find_run, load_log

def table(headers, rows):
    display(Markdown('| ' + ' | '.join(headers) + ' |\n| ' + ' | '.join('---' for _ in headers) + ' |\n' + '\n'.join('| ' + ' | '.join(map(str, row)) + ' |' for row in rows)))

log = load_log()
print('Loaded', len(log), 'benchmark rows from', ROOT / 'bench' / 'bench_log.csv')

Loaded 13 benchmark rows from G:\VIT\Intern\submition\partB\bench\bench_log.csv


## B1 — KV cache

$28 \\times 8 \\times 128 \\times 2_{K+V} \\times 2_{fp16} = 114{,}688$ bytes/token.

In [2]:
s = b1_summary()
table(['quantity', 'value'], [
    ['KV bytes/token', f"{s['kv_bytes_per_token']:,}"],
    ['usable GPU', f"{s['usable_gpu_gb']:.2f} GB"],
    ['KV budget', f"{s['kv_budget_gb']:.2f} GB"],
    ['max KV tokens', f"{s['max_kv_tokens']:.2f}"],
    ['max full 4096 sequences', f"{s['max_concurrent_4096']:.3f} → {s['max_concurrent_4096_floor']}"]
])
table(['run', 'predicted KV util', 'logged KV util', 'preempted'], [[f"{'long' if r.is_long else 'short'} bs{r.batch_size}", f"{r.predicted_kv_util:.3f}", f"{r.kv_cache_util:.2f}", r.preempted_seqs] for r in log])

| quantity | value |
| --- | --- |
| KV bytes/token | 114,688 |
| usable GPU | 22.08 GB |
| KV budget | 12.08 GB |
| max KV tokens | 105329.24 |
| max full 4096 sequences | 25.715 → 25 |

| run | predicted KV util | logged KV util | preempted |
| --- | --- | --- | --- |
| short bs1 | 0.007 | 0.01 | 0 |
| short bs2 | 0.015 | 0.01 | 0 |
| short bs4 | 0.029 | 0.03 | 0 |
| short bs8 | 0.058 | 0.06 | 0 |
| short bs16 | 0.117 | 0.12 | 0 |
| short bs32 | 0.233 | 0.23 | 0 |
| short bs64 | 0.467 | 0.47 | 0 |
| long bs4 | 0.156 | 0.16 | 0 |
| long bs8 | 0.311 | 0.31 | 0 |
| long bs16 | 0.622 | 0.62 | 0 |
| long bs24 | 0.933 | 0.93 | 0 |
| long bs32 | 1.244 | 0.97 | 7 |
| long bs48 | 1.867 | 0.97 | 23 |

## B2 and B3 — Anomaly and honest goodput

Long-context throughput peaks at batch 24, then falls when predicted KV demand exceeds the pool and preemption appears. A 24-sequence active cap should avoid preemption, but latency for a queued 48-request burst is unmeasured.

In [3]:
long = [r for r in log if r.is_long]
table(['batch', 'reported tok/s', 'generated tok/s', 'wall s', 'KV util', 'preempted'], [[r.batch_size, f"{r.reported_tok_s:.1f}", f"{r.gen_goodput:.1f}", f"{r.wall_clock_s:.2f}", f"{r.kv_cache_util:.2f}", r.preempted_seqs] for r in long])
r24 = find_run(24, 3584, log)
way1 = r24.num_requests * r24.gen_len / r24.wall_clock_s
way2 = r24.reported_tok_s * r24.gen_len / r24.seq_len
print(f'B3 equivalent derivations: {way1:.2f} and {way2:.2f} generated tok/s')

| batch | reported tok/s | generated tok/s | wall s | KV util | preempted |
| --- | --- | --- | --- | --- | --- |
| 4 | 565.4 | 70.7 | 28.98 | 0.16 | 0 |
| 8 | 902.6 | 112.8 | 36.30 | 0.31 | 0 |
| 16 | 1311.4 | 163.9 | 49.97 | 0.62 | 0 |
| 24 | 1607.4 | 200.9 | 61.16 | 0.93 | 0 |
| 32 | 1384.0 | 173.0 | 94.71 | 0.97 | 7 |
| 48 | 1298.5 | 162.3 | 151.41 | 0.97 | 23 |

B3 equivalent derivations: 200.92 and 200.93 generated tok/s


## B4 — Production counter

Monitor the delta of the deployed vLLM scheduler preemption counter (commonly `vllm:num_preemptions_total`; verify the version-specific name). Expect zero below the long-context knee and a positive delta above it. The CSV's 7/23 values count unique sequences preempted at least once, not necessarily total preemption events.